<a href="https://www.kaggle.com/code/ujwal960/04-resnet18-ipynb?scriptVersionId=312857364" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [18]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


In [21]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import numpy as np
from transformers import ViTForImageClassification

print("All libraries imported!")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

All libraries imported!
Using device: cuda


In [22]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset_full = datasets.GTSRB(root='./data', split='train', transform=transform_train, download=True)
test_dataset = datasets.GTSRB(root='./data', split='test', transform=transform_test, download=True)

train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 23976
Validation samples: 2664
Test samples: 12630


In [23]:
# Load pretrained ResNet18
model_resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model_resnet.fc = nn.Linear(model_resnet.fc.in_features, 43)
model_resnet = model_resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_resnet.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

best_val_acc = 0
patience = 3
patience_counter = 0

for epoch in range(10):
    # Training
    model_resnet.train()
    correct, total = 0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/10"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100 * correct / total
    
    # Validation
    model_resnet.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_resnet(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model_resnet.state_dict(), 'resnet18_best.pth')
        print(f"  → Best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    scheduler.step()

print(f"\nBest Val Accuracy: {best_val_acc:.2f}%")
print("ResNet18 Training complete!")

Epoch 1/10: 100%|██████████| 375/375 [01:15<00:00,  5.00it/s]


Epoch 1: Train Acc=88.15%, Val Acc=98.87%
  → Best model saved!


Epoch 2/10: 100%|██████████| 375/375 [01:14<00:00,  5.01it/s]


Epoch 2: Train Acc=99.43%, Val Acc=99.62%
  → Best model saved!


Epoch 3/10: 100%|██████████| 375/375 [01:14<00:00,  5.01it/s]


...

Epoch 4: Train Acc=99.98%, Val Acc=100.00%
  → Best model saved!


Epoch 5/10: 100%|██████████| 375/375 [01:14<00:00,  5.01it/s]


Epoch 5: Train Acc=99.96%, Val Acc=99.92%


Epoch 6/10:  14%|█▍        | 53/375 [00:10<01:06,  4.82it/s]


KeyboardInterrupt: 

In [24]:
model_resnet.load_state_dict(torch.load('resnet18_best.pth'))
model_resnet.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing ResNet18"):
        images, labels = images.to(device), labels.to(device)
        outputs = model_resnet(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"ResNet18 Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"ResNet18 Macro F1: {test_f1:.4f}")
torch.save(model_resnet.state_dict(), 'resnet18_final.pth')
print("ResNet18 saved!")

Testing ResNet18: 100%|██████████| 198/198 [00:15<00:00, 13.16it/s]

ResNet18 Test Accuracy: 96.83%
ResNet18 Macro F1: 0.9253
ResNet18 saved!


In [ ]:
# Experiment: Added dropout (0.3) to reduce overfitting
# Hypothesis: Dropout would improve generalization on test set
# Result: Test Accuracy = 95.80% - worse than without dropout
# Conclusion: Dropout hurt performance here.

In [25]:
# Retrain ResNet18 fresh - 3 epochs with dropout to reduce overfitting
model_resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model_resnet.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(512, 43)
)
model_resnet = model_resnet.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_resnet.parameters(), lr=0.0001)

best_val_acc = 0
for epoch in range(3):
    # Training
    model_resnet.train()
    correct, total = 0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/3"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100 * correct / total
    
    # Validation
    model_resnet.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_resnet(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    print(f"Epoch {epoch+1}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_resnet.state_dict(), 'resnet18_best.pth')
        print(f"  → Best model saved!")

print(f"\nBest Val Accuracy: {best_val_acc:.2f}%")

# Evaluate on test set
model_resnet.load_state_dict(torch.load('resnet18_best.pth'))
model_resnet.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = model_resnet(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"\nResNet18 Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"ResNet18 Macro F1: {test_f1:.4f}")
torch.save(model_resnet.state_dict(), 'resnet18_final.pth')
print("ResNet18 saved!")

Epoch 1/3: 100%|██████████| 375/375 [01:14<00:00,  5.03it/s]


Epoch 1: Train Acc=86.61%, Val Acc=98.69%
  → Best model saved!


Epoch 2/3: 100%|██████████| 375/375 [01:14<00:00,  5.02it/s]


Epoch 2: Train Acc=99.10%, Val Acc=99.47%
  → Best model saved!


Epoch 3/3: 100%|██████████| 375/375 [01:14<00:00,  5.02it/s]


Epoch 3: Train Acc=99.63%, Val Acc=99.66%
  → Best model saved!

Best Val Accuracy: 99.66%


Testing: 100%|██████████| 198/198 [00:14<00:00, 13.23it/s]


ResNet18 Test Accuracy: 95.80%
ResNet18 Macro F1: 0.9031
ResNet18 saved!


In [ ]:
# ============================================
# RESNET18 FINAL RESULTS SUMMARY
# ============================================
# 
# Three experiments conducted to find optimal configuration:
#
# Experiment 1: 4 epochs, no dropout
#   → Train Acc = 99.98%, Val Acc = 100.00%
#   → Test Accuracy = 96.83%, Macro F1 = 0.9253
#   → Mild overfitting (3.17% gap)
#
# Experiment 2: 3 epochs, dropout=0.3
#   → Train Acc = 99.63%, Val Acc = 99.66%
#   → Test Accuracy = 95.80%, Macro F1 = 0.9031
#   → Dropout did not improve generalization
#
# Experiment 3: 3 epochs, no dropout (clean notebook)
#   → Test Accuracy = 96.03%, Macro F1 = 0.9093
#   → Slightly better than dropout but worse than 4 epochs
#
# CONCLUSION:
#   Best configuration = 4 epochs, no dropout
#   Final ResNet18 Test Accuracy = 96.83%
#   Final ResNet18 Macro F1 = 0.9253
# ============================================